In [ ]:
!pip install sentence-transformers pypdf pandas scikit-learn

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving kaviya_baskaran_resume.pdf to kaviya_baskaran_resume (1).pdf
Saving Mirudhula_Resume.pdf to Mirudhula_Resume (1).pdf
Saving Pavithra_N.pdf to Pavithra_N (1).pdf
Saving Shake_Ashiya.pdf to Shake_Ashiya (1).pdf


In [ ]:
from pypdf import PdfReader

def extract_text(pdf_file):

    text = ""

    reader = PdfReader(pdf_file)

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text

    return text

In [ ]:
resume_data = {}

for filename in uploaded.keys():

    resume_text = extract_text(filename)

    resume_data[filename] = resume_text

print("Total Resumes:", len(resume_data))

Total Resumes: 4


In [ ]:
jd = """
Looking for Java Developer

Skills:
Java
Spring Boot
REST API
MySQL
DSA
Git
"""

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

results = []

jd_embedding = model.encode(jd)

for name, text in resume_data.items():

    resume_embedding = model.encode(text)

    similarity = cosine_similarity(
        [jd_embedding],
        [resume_embedding]
    )[0][0]

    match_percent = round(
        similarity * 100,
        2
    )

    results.append(
        [name, match_percent]
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
import pandas as pd

df = pd.DataFrame(
    results,
    columns=[
        "Candidate",
        "Match %"
    ]
)

df = df.sort_values(
    by="Match %",
    ascending=False
)

df.reset_index(drop=True, inplace=True)

df.index = df.index + 1
df.index.name = "Rank"

display(df)

,Candidate,Match %
Rank,,
1,Mirudhula_Resume (1).pdf,47.130001
2,Pavithra_N (1).pdf,42.119999
3,Shake_Ashiya (1).pdf,39.639999
4,kaviya_baskaran_resume (1).pdf,27.910000


In [ ]:
import google.generativeai as genai

genai.configure(api_key="AQ.Ab8RN6LMPMLN_uJD2j-oEhlJAAznumdoBSSy_7UALv0SPudd5w")

gemini_model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

In [ ]:
all_resumes = ""

for name, text in resume_data.items():

    all_resumes += f"\n\nCandidate: {name}\n{text}"

In [ ]:
question = input("Ask your question: ")

Ask your question: who is ml engineer?


In [ ]:
prompt = f"""
You are an HR recruiter.

Job Description:
{jd}

Candidate Rankings:
{df.to_string()}

Candidate Resumes:
{all_resumes}

Question:
{question}

Answer with:
1. Candidate Name
2. Evidence from Resume
3. Match Percentage
"""

In [ ]:
response = gemini_model.generate_content(
    prompt
)

print(response.text)

1.  **Candidate Name:** Mirudhula R

2.  **Evidence from Resume:**
    *   **SUMMARY:** "Interested in software development, **machine learning**, and building practical real-world solutions through strong analytical and programming abilities."
    *   **PROJECTS:**
        *   "**AI Based Inclusive Form Assistant**": Developed an "intelligent multilingual chatbot system that extracts questions from scanned government forms (PDF/ Image) using OCR and **rule-based NLP**."
        *   "**Smart Meeting Transcript & Summary System**": Developed an "**AI-powered web application** that converts meeting audio into text transcripts using **speech recognition techniques**. Implemented **Natural Language Processing (NLP)** to generate structured meeting summaries."
    *   **COURSES:** "Machine Learning Specialization, Coursera – offered by Stanford University. Completed the course in the Machine Learning Specialization focused on supervised learning techniques. Gained practical understanding of